In [1]:
from pymongo import MongoClient
mongo_url = "mongodb://mongodb:65530"
# Connect to MongoDB using the URL
client = MongoClient(mongo_url)
# Access your database and collection
db = client["oscb"]

pp_results_collection = db.get_collection("pp_results")

In [2]:
result = list(pp_results_collection.find({"zarr_path": {"$exists": True}}, {'process_id': 1, 'adata_path': 1}))
result

[{'_id': ObjectId('68a3f6171b6ee93a6ab29d35'),
  'process_id': 'ee8512f99a1ec9e823734e40edd14167',
  'adata_path': '/usr/src/app/storage/Benchmarks/dropletLung_1755575440627/QC/droplet_Lung_Seurat.h5ad'},
 {'_id': ObjectId('68a5dd8a112dbe0b5f3df8b4'),
  'process_id': 'b2866d400e21677e5a8ac383671c3899',
  'adata_path': '/usr/src/app/storage/Benchmarks/facsBladder_1755700462145/QC/facs_Bladder_Seurat.h5ad'},
 {'_id': ObjectId('68a60215112dbe0b5f3e42ae'),
  'process_id': '94415f362a8931e9c398a47eb32a563a',
  'adata_path': '/usr/src/app/storage/Benchmarks/facsLimbMuscle_1755709856736/QC/facs_Limb_Muscle_Seurat.h5ad'},
 {'_id': ObjectId('68a62c7f112dbe0b5f3e9829'),
  'process_id': '8b879ecd2f2e6c5c7a3e5bd1f976daf1',
  'adata_path': '/usr/src/app/storage/Benchmarks/facsThymus_1755720675174/QC/facs_Thymus_Seurat.h5ad'},
 {'_id': ObjectId('68aa1890d4f3c18ef9f20a92'),
  'process_id': 'd4747966ec5dfd1ce14c8f8e73684eb5',
  'adata_path': '/usr/src/app/storage/Benchmarks/TSepithelial_1755977757845/

In [3]:
import numpy as np
import jax
import jax.numpy as jnp
from typing import Optional, Union
import pandas as pd
import scipy.sparse as sp_sparse
from scipy.sparse import csr_matrix
import h5py

try:
    from anndata._core.sparse_dataset import SparseDataset
except ImportError:
    # anndata >= 0.10.0
    from anndata._core.sparse_dataset import (
        BaseCompressedSparseDataset as SparseDataset,
    )

def is_normalized(expression_matrix, min_genes=200):
    if (not isinstance(expression_matrix, np.ndarray)):
        expression_matrix = expression_matrix.toarray()

    if np.min(expression_matrix) < 0 or np.max(expression_matrix) < min_genes:
        return True
    else:
        return False
        

def check_nonnegative_integers(
    data: Union[pd.DataFrame, np.ndarray, sp_sparse.spmatrix, h5py.Dataset],
    n_to_check: int = 20,
):
    """Approximately checks values of data to ensure it is count data."""
    # for backed anndata
    if isinstance(data, h5py.Dataset) or isinstance(data, SparseDataset):
        data = data[:100]

    if isinstance(data, np.ndarray):
        data = data
    elif issubclass(type(data), sp_sparse.spmatrix):
        data = data.data
    elif isinstance(data, pd.DataFrame):
        data = data.to_numpy()
    else:
        raise TypeError("data type not understood")

    ret = True
    if len(data) != 0:
        inds = np.random.choice(len(data), size=(n_to_check,))
        check = jax.device_put(data.flat[inds], device=jax.devices("cpu")[0])
        negative, non_integer = _is_not_count_val(check)
        ret = not (negative or non_integer)
    return ret


@jax.jit
def _is_not_count_val(data: jnp.ndarray):
    negative = jnp.any(data < 0)
    non_integer = jnp.any(data % 1 != 0)

    return negative, non_integer


/opt/conda/lib/python3.11/site-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.6.2 is installed, but it is not compatible with the installed jaxlib version 0.6.1, so it will not be used.
  warnings.warn(


In [4]:
def save_zarr(adata, adata_path, n_hvg=50, layer=None, min_genes=200):
    zarr_output = adata_path.replace('storage/', 'storage/zarr/').replace('.h5ad', '.zarr')

    unique_cell_labels = []
    obs_cols = []

    if 'leiden' in adata.obs.columns:
        obs_cols.append('leiden')
    # Retrieve unique cell type labels
    try:
        with open('uniqueCellLabels.json', 'r', encoding='utf-8') as file:
            unique_cell_labels = json.load(file)
    except FileNotFoundError:
        print("The specified JSON file was not found.")
    except json.JSONDecodeError:
        print("Error decoding JSON. Ensure the file contains a valid list.")
    # Add cell type annotations to obsSets
    if len(unique_cell_labels) > 0:
        for label in unique_cell_labels:
            if label in adata.obs.columns:
                obs_cols.append(label)

    if layer is not None and layer in adata.layers.keys():
        adata.X = adata.layers[layer]
    elif not is_normalized(adata.X, min_genes) or check_nonnegative_integers(adata.X):
        if "scale.data" in adata.layers.keys():
            adata.X = adata.layers["scale.data"]
        elif "normalized_X" in adata.layers.keys():
            adata.X = adata.layers["normalized_X"]
        else:
            # Perform normalization
            sc.pp.normalize_total(adata, inplace=True)
            sc.pp.log1p(adata)
    
    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=n_hvg)
    # Get the highly variable gene matrix as a plain NumPy array
    X_hvg_arr = adata[:, adata.var['highly_variable']].X.toarray()
    X_hvg_index = adata[:, adata.var['highly_variable']].var.copy().index
    # Get the highly variable gene matrix as a plain NumPy array
    X_hvg_arr = adata[:, adata.var['highly_variable']].X.toarray()
    X_hvg_index = adata[:, adata.var['highly_variable']].var.copy().index

    # Perform average linkage hierarchical clustering on along the genes axis of the array
    Z = scipy.cluster.hierarchy.linkage(X_hvg_arr.T, method="average", optimal_ordering=True)

    # Get the hierarchy-based ordering of genes.
    num_genes = adata.var.shape[0]
    highly_var_index_ordering = scipy.cluster.hierarchy.leaves_list(Z)
    highly_var_genes = X_hvg_index.values[highly_var_index_ordering].tolist()

    all_genes = adata.var.index.values.tolist()
    not_var_genes = adata.var.loc[~adata.var['highly_variable']].index.values.tolist()

    def get_orig_index(gene_id):
        return all_genes.index(gene_id)
    var_index_ordering = list(map(get_orig_index, highly_var_genes)) + list(map(get_orig_index, not_var_genes))
    adata = adata[:, var_index_ordering].copy()
    adata.obsm['X_hvg'] = adata[:, adata.var['highly_variable']].X.copy()

    adata = optimize_adata(
        adata,
        obs_cols = obs_cols, # Add your "hue" columns here
        obsm_keys = ["X_umap"],             # Add your UMAP key
    )

    # Write out to Zarr
    # Vitessce expects a specific hierarchy. This helper makes it compatible.
    adata.write_zarr(zarr_output, chunks=(adata.n_obs, adata.n_vars))
    
    return zarr_output

In [ ]:
import scanpy as sc
import json
import scipy.cluster
from vitessce.data_utils import optimize_adata

for r in result:
    adata_path=r['adata_path']
    process_id=r['process_id']
    if adata_path.endswith(".h5ad"):
        try:
            print(adata_path)
            adata = sc.read_h5ad(adata_path)
            zarr_path = save_zarr(adata, adata_path, n_hvg=50, layer=None, min_genes=200)
            print("zarr is saved for " + process_id + " at: " + zarr_path)
            pp_results_collection.update_one({'process_id': process_id}, {'$set': {"zarr_path": zarr_output}})
            print("MongoDB record is updated for " + process_id)
        except Exception as e:
            print(e)

/opt/conda/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(


/usr/src/app/storage/Benchmarks/dropletLung_1755575440627/QC/droplet_Lung_Seurat.h5ad


ERROR:2025-12-29 00:32:00,582:jax._src.xla_bridge:444: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/jax/_src/xla_bridge.py", line 442, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/opt/conda/lib/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 324, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/opt/conda/lib/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 281, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load the CUDA libraries.


zarr is saved for ee8512f99a1ec9e823734e40edd14167 at: /usr/src/app/storage/Benchmarks/dropletLung_1755575440627/QC/droplet_Lung_Seurat.zarr
/usr/src/app/storage/Benchmarks/facsBladder_1755700462145/QC/facs_Bladder_Seurat.h5ad
zarr is saved for b2866d400e21677e5a8ac383671c3899 at: /usr/src/app/storage/Benchmarks/facsBladder_1755700462145/QC/facs_Bladder_Seurat.zarr
/usr/src/app/storage/Benchmarks/facsLimbMuscle_1755709856736/QC/facs_Limb_Muscle_Seurat.h5ad
zarr is saved for 94415f362a8931e9c398a47eb32a563a at: /usr/src/app/storage/Benchmarks/facsLimbMuscle_1755709856736/QC/facs_Limb_Muscle_Seurat.zarr
/usr/src/app/storage/Benchmarks/facsThymus_1755720675174/QC/facs_Thymus_Seurat.h5ad
zarr is saved for 8b879ecd2f2e6c5c7a3e5bd1f976daf1 at: /usr/src/app/storage/Benchmarks/facsThymus_1755720675174/QC/facs_Thymus_Seurat.zarr
/usr/src/app/storage/Benchmarks/TSepithelial_1755977757845/QC/TS_epithelial.h5ad
zarr is saved for d4747966ec5dfd1ce14c8f8e73684eb5 at: /usr/src/app/storage/Benchmarks/

adata.X seems to be already log-transformed.


...

zarr is saved for 12b2766c68e78d20633550395af4645d at: /usr/src/app/storage/Benchmarks/PBMC1K_1757469243117/QC/PBMC_1K_scanpy.zarr
/usr/src/app/storage/Benchmarks/Lung_1757470755879/QC/Lung_scanpy.h5ad
zarr is saved for 6d2e31ebd445d8b967fee4a7ddcc578d at: /usr/src/app/storage/Benchmarks/Lung_1757470755879/QC/Lung_scanpy.zarr
/usr/src/app/storage/Benchmarks/Mousebrainatlas_1757556315095/QC/Mouse_brain_atlas_scanpy.h5ad


adata.X seems to be already log-transformed.


zarr is saved for 35be43e4ccba6617aae13cbb941b4967 at: /usr/src/app/storage/Benchmarks/Mousebrainatlas_1757556315095/QC/Mouse_brain_atlas_scanpy.zarr
/usr/src/app/storage/Benchmarks/Triple-negative-breast-cancer-atlas_1757635543985/QC/Triple_negative_breast_cancer_atlas_scanpy.h5ad


adata.X seems to be already log-transformed.


zarr is saved for 3a89edbc8aa1ff4ca959601ad4365aba at: /usr/src/app/storage/Benchmarks/Triple-negative-breast-cancer-atlas_1757635543985/QC/Triple_negative_breast_cancer_atlas_scanpy.zarr
/usr/src/app/storage/Benchmarks/HypoMap_1760764591725/QC/HypoMap.h5ad


In [ ]:
import scanpy as sc
import json
from anndata import AnnData

unique_cell_labels = []

# Retrieve unique cell type labels
try:
    with open('uniqueCellLabels.json', 'r', encoding='utf-8') as file:
        unique_cell_labels = json.load(file)
        print(unique_cell_labels)
except FileNotFoundError:
    print("The specified JSON file was not found.")
except json.JSONDecodeError:
    print("Error decoding JSON. Ensure the file contains a valid list.")

for r in result:
    adata = None
    zarr_path=None
    initialFeatureFilterPath="var/highly_variable"
    obsEmbedding="obsm/X_umap"
    obsSets=[{"name": "Cluster", "path": "obs/leiden"}]
    adata_path=r['adata_path']
    process_id=r['process_id']
    if adata_path.endswith(".h5ad"):
        try:
            print(adata_path)
            adata = sc.read_h5ad(adata_path)
            # Add cell type annotations to obsSets
            if len(unique_cell_labels) > 0:
                for label in unique_cell_labels:
                    if label in adata.obs.columns:
                        obsSets.append({"name": label, "path": "obs/" + label})
            zarr_output = adata_path.replace('.h5ad', '.zarr')
            adata.write_zarr(zarr_output)
            print("zarr is saved for " + process_id)
    
            pp_results_collection.update_one({'process_id': process_id}, {'$set': {"zarr_path": zarr_output, "initialFeatureFilterPath": initialFeatureFilterPath, "obsEmbedding": obsEmbedding, "obsSets": obsSets}})
            print("MongoDB record is updated for " + process_id)
        except Exception as e:
            print(e)
            
        

In [ ]:
bdata = sc.read_h5ad('/usr/src/app/storage/Benchmarks/TSLung_1755833091663/QC/TS_Lung_scanpy.h5ad')
bdata

In [ ]:
bdata.X.todense()

In [ ]:
is_normalized(adata.X, 200)

In [ ]:
check_nonnegative_integers(adata.X)

In [ ]:
adata.X = adata.layers['raw_counts'].copy()

In [ ]:
not is_normalized(adata.X, 200) or check_nonnegative_integers(adata.X)

In [ ]:
import scanpy as sc
import json
from anndata import AnnData
from vitessce.data_utils import optimize_adata

In [ ]:
adata_path='/usr/src/app/storage/Benchmarks/facsKidney_1755709836242/QC/facs_Kidney_Seurat.h5ad'

adata = sc.read_h5ad(adata_path)

adata

In [ ]:
adata.X

In [ ]:
adata.layers['raw_counts']

In [ ]:
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=50)

In [ ]:
# Get the highly variable gene matrix as a plain NumPy array
X_hvg_arr = adata[:, adata.var['highly_variable']].X.toarray()
X_hvg_index = adata[:, adata.var['highly_variable']].var.copy().index

In [ ]:
import scipy.cluster

# Get the highly variable gene matrix as a plain NumPy array
X_hvg_arr = adata[:, adata.var['highly_variable']].X.toarray()
X_hvg_index = adata[:, adata.var['highly_variable']].var.copy().index

# Perform average linkage hierarchical clustering on along the genes axis of the array
Z = scipy.cluster.hierarchy.linkage(X_hvg_arr.T, method="average", optimal_ordering=True)

# Get the hierarchy-based ordering of genes.
num_genes = adata.var.shape[0]
highly_var_index_ordering = scipy.cluster.hierarchy.leaves_list(Z)
highly_var_genes = X_hvg_index.values[highly_var_index_ordering].tolist()

all_genes = adata.var.index.values.tolist()
not_var_genes = adata.var.loc[~adata.var['highly_variable']].index.values.tolist()

def get_orig_index(gene_id):
    return all_genes.index(gene_id)
var_index_ordering = list(map(get_orig_index, highly_var_genes)) + list(map(get_orig_index, not_var_genes))

In [ ]:
adata = adata[:, var_index_ordering].copy()
adata

In [ ]:
adata.obsm['X_hvg'] = adata[:, adata.var['highly_variable']].X.copy()

In [ ]:
adata

In [ ]:
adata.obsm['X_hvg'].shape

In [ ]:

obs_cols=["leiden", "cell_ontology_class"]

zarr_output = adata_path.replace('.h5ad', '.zarr')
adata = optimize_adata(
    adata,
    obs_cols=obs_cols, # Add your "hue" columns here
    obsm_keys=["X_umap"],             # Add your UMAP key
)

# Vitessce expects a specific hierarchy. This helper makes it compatible.
adata.write_zarr(zarr_output, chunks=(adata.n_obs, adata.n_vars))

In [ ]:
adata

In [ ]:
adata.var.highly_variable